In [3]:
#!/usr/bin/env python
"""Interactive UMAP/t-SNE explorer with 2D/3D, HDBSCAN, manual clusters, ID projection and exports."""

###################################### imports
from datetime import datetime
import os
import resource
import time
import traceback
from pathlib import Path

import hdbscan
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import clear_output, display
from ipywidgets import Button, Checkbox, ColorPicker, Dropdown, HBox, IntSlider, Output, Text, VBox, Layout
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyprojroot import here

###################################### constants
BASE_DIR = "data/dim_red"
SEED = 42
MARKER_SIZE = 2
MARKER_SIZE_3D = 2
DEFAULT_COLOR = "#A5A5A5"
ALPHA = 0.75
ALPHA_INACTIVE = 0.025
PALETTE_NATURE = ["#e86c60", "#96c973", "#e9c46a", "#2a9d8f", "#4F6F7C", "#264653"]
MIN_CLUSTER_SIZE = 800
MAX_CLUSTERS = 15
FIGURE_WIDTH = 1100
FIGURE_HEIGHT = 620
SQUARE_SIZE = 8.0
ZOOM_FACTOR_UMAP = 0.9
ZOOM_FACTOR_TSNE = 1.10
TICK_FONT_SIZE = 10
TICK_FONT_FAMILY = "DejaVu Sans"
TICK_DECIMALS = 1
N_TICKS = 6
LEGEND_FONT_SIZE = 9
TITLE_FONT_SIZE = 14
DPI = 300
LEGEND_NCOL = 4
LINE_COLOR = "black"
LINE_WIDTH = 0.8

###################################### setup
start_time = time.time()
PROJECT_ROOT = Path(here()).expanduser().resolve()
WORKING_DIR = Path.cwd().expanduser().resolve()


def _unique_paths(paths):
    result = []
    seen = set()
    for item in paths:
        if item is None:
            continue
        path = Path(item).expanduser().resolve()
        if path not in seen:
            seen.add(path)
            result.append(path)
    return result


def _paired_embedding_count(directory):
    """Count complete UMAP/t-SNE pairs in one directory; never inspect config files."""
    count = 0
    for dim in ("2D", "3D"):
        for path in directory.glob(f"*_{dim}_umap.npz"):
            basename = path.name[: -len(f"_{dim}_umap.npz")]
            if (directory / f"{basename}_{dim}_tsne.npz").is_file():
                count += 1
    return count


def _find_embedding_dir():
    """Find a directory containing paired NPZ embeddings, without touching config files."""
    override = os.environ.get("DIMRED_BASE_DIR")
    if override:
        path = Path(override).expanduser()
        if not path.is_absolute():
            path = WORKING_DIR / path
        if not path.is_dir():
            raise FileNotFoundError(f"DIMRED_BASE_DIR points to a missing directory: {path}")
        if _paired_embedding_count(path) == 0:
            raise FileNotFoundError(
                f"No paired UMAP/t-SNE NPZ files found in {path}. Expected matching names like "
                "<dataset>_2D_umap.npz and <dataset>_2D_tsne.npz."
            )
        return path.resolve()

    # First try the conventional locations relative to both notebook cwd and project root.
    relative = Path(BASE_DIR)
    candidates = _unique_paths([
        PROJECT_ROOT / relative,
        WORKING_DIR / relative,
        PROJECT_ROOT / relative.name,
        WORKING_DIR / relative.name,
    ])
    for candidate in candidates:
        if candidate.is_dir() and _paired_embedding_count(candidate) > 0:
            return candidate

    # If the directory is named differently, search for NPZ embedding pairs under the
    # project/notebook roots. This scan is restricted to directories and .npz filenames;
    # it never opens or searches configuration files.
    excluded = {".git", ".venv", "venv", "node_modules", "__pycache__", ".ipynb_checkpoints"}
    found = []
    for root in _unique_paths([PROJECT_ROOT, WORKING_DIR]):
        if not root.is_dir():
            continue
        try:
            for path in root.rglob("*.npz"):
                if any(part in excluded for part in path.parts):
                    continue
                count = _paired_embedding_count(path.parent)
                if count:
                    found.append((count, path.parent.resolve()))
        except (PermissionError, OSError):
            continue
    if not found:
        return None
    # Prefer the directory with the most complete pairs; tie-break deterministically.
    found.sort(key=lambda item: (-item[0], str(item[1])))
    return found[0][1]


BASE_PATH = _find_embedding_dir()
if BASE_PATH is None:
    raise FileNotFoundError(
        "Could not find any directory containing paired UMAP/t-SNE embedding NPZ files. "
        "Set DIMRED_BASE_DIR to the exact directory containing files named like "
        "<dataset>_2D_umap.npz and <dataset>_2D_tsne.npz. "
        f"Searched beneath project root {PROJECT_ROOT} and notebook working directory {WORKING_DIR}. "
        "No configuration files were accessed."
    )

print(f"[PYTHON-INFO] Project root: {PROJECT_ROOT}")
print(f"[PYTHON-INFO] Embeddings: {BASE_PATH}")

# Dataset names are inferred exclusively from paired embedding filenames.
def _has_pair(name, dim):
    return all((BASE_PATH / f"{name}_{dim}_{method}.npz").is_file() for method in ("umap", "tsne"))

valid_options_all = []
for dim in ("2D", "3D"):
    for path in BASE_PATH.glob(f"*_{dim}_umap.npz"):
        basename = path.name[: -len(f"_{dim}_umap.npz")]
        if _has_pair(basename, dim):
            valid_options_all.append(f"{basename}_{dim}")
valid_options_all = sorted(set(valid_options_all))
if not valid_options_all:
    raise FileNotFoundError(
        f"No paired UMAP/t-SNE embedding files found in {BASE_PATH}. Expected filenames like "
        "<dataset>_2D_umap.npz and <dataset>_2D_tsne.npz."
    )

# Without a metadata/config file, each embedding dataset is treated as a single label.
label_to_species = {}
groups_dict = {}

###################################### widgets
select_dropdown = Dropdown(options=[""] + valid_options_all, description="Dataset:", layout=Layout(width="300px"))
color_source_dropdown = Dropdown(options=["(Self)"] + valid_options_all, value="(Self)", description="Color by:", layout=Layout(width="300px"))
reset_angles_btn = Button(description="Reset view", button_style="warning")
checklist_box = HBox([])
mcs_slider = IntSlider(value=MIN_CLUSTER_SIZE, min=100, max=10000, step=10, description="MinCS:", continuous_update=False, layout=Layout(width="320px"))
fit_on_dropdown = Dropdown(options=["umap", "tsne"], value="umap", description="Fit on:")
fit_btn = Button(description="Fit HDBSCAN", button_style="primary")
cluster_name = Text(value="NewCluster", description="Name:", layout=Layout(width="210px"))
cluster_color = ColorPicker(value="#E7298A", description="Color:")
assign_btn = Button(description="Assign selected", button_style="primary")
rename_dropdown = Dropdown(options=[], description="Rename:", layout=Layout(width="210px"))
rename_text = Text(value="", description="To:", layout=Layout(width="160px"))
rename_btn = Button(description="Rename", button_style="info")
reset_btn = Button(description="Reset clusters", button_style="danger")
delete_btn = Button(description="Delete cluster", button_style="danger")
plot_fp_text = Text(value="viz/dimred", description="Export dir:", layout=Layout(width="340px"))
export_tsv_btn = Button(description="Export TSV", button_style="info")
export_plot_btn = Button(description="Export Plot", button_style="info")
run_all_btn = Button(description="Run All", button_style="success")
output_area = Output()
plot_container = VBox([])
state = {"ids_umap": None, "umap_coords": None, "tsne_coords": None, "cluster_labels": None,
         "cluster_info": {}, "filtered_mask": None, "current_select": None, "label_source": None,
         "checkboxes": {}, "fig_widget": None, "selected_indices": [], "last_selection_source": "umap",
         "display_title": "", "dim": "2D"}

###################################### data and clustering
def load_npz(basename, method="umap", dim="2D"):
    path = BASE_PATH / f"{basename}_{dim}_{method}.npz"
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        if "ids" not in data.files:
            raise ValueError(f"'ids' array missing from {path}")
        ids = np.asarray(data["ids"])
        keys = [key for key in data.files if key != "ids"]
        if not keys:
            raise ValueError(f"No embedding array found in {path}")
        coords = np.asarray(data[keys[0]], dtype=np.float64)
    expected = 3 if dim == "3D" else 2
    if coords.ndim != 2 or coords.shape[1] != expected or len(ids) != len(coords):
        raise ValueError(f"Invalid IDs/embedding dimensions in {path}: ids={len(ids)}, coords={coords.shape}")
    if pd.Index(ids).has_duplicates:
        raise ValueError(f"Duplicate IDs in {path}; ID-based projection would be ambiguous")
    return ids, coords

def align_embedding(reference_ids, ids, coords, source_name):
    lookup = {value: i for i, value in enumerate(ids.tolist())}
    missing = [value for value in reference_ids.tolist() if value not in lookup]
    if missing:
        raise ValueError(f"{source_name} is missing {len(missing)} IDs present in the UMAP file")
    return coords[np.fromiter((lookup[value] for value in reference_ids.tolist()), dtype=int, count=len(reference_ids))]

def fit_hdbscan_auto(coords, min_cluster_size):
    coords = np.asarray(coords, dtype=np.float64)
    mcs = int(min_cluster_size)
    if len(coords) < mcs:
        raise ValueError(f"Only {len(coords)} points available; MinCS is {mcs}")
    raw = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=min(10, mcs), metric="euclidean", gen_min_span_tree=True, prediction_data=True).fit_predict(coords)
    valid = raw[raw >= 0]
    if not len(valid):
        return np.zeros(len(coords), dtype=int), 0
    unique, counts = np.unique(valid, return_counts=True)
    ordered = unique[np.argsort(-counts)]
    mapping = {int(old): rank + 1 for rank, old in enumerate(ordered[:MAX_CLUSTERS])}
    return np.asarray([mapping.get(int(x), 0) if x >= 0 else 0 for x in raw], dtype=int), len(mapping)

def next_cluster_id():
    return max((int(cid) for cid in state["cluster_info"] if int(cid) != 0), default=0) + 1

def cluster_color_for(cid):
    return state["cluster_info"].get(int(cid), {"color": DEFAULT_COLOR})["color"]

def cluster_name_for(cid):
    return state["cluster_info"].get(int(cid), {"name": "unassigned"})["name"]

###################################### plot refresh
def update_checklist(labels):
    state["checkboxes"] = {}
    boxes = []
    for label in labels:
        cb = Checkbox(value=True, description=label, indent=False)
        cb.observe(lambda change: update_visibility(), names="value")
        state["checkboxes"][label] = cb
        boxes.append(cb)
    checklist_box.children = boxes

def refresh_cluster_dropdown():
    options = [f"{cid}: {info['name']}" for cid, info in sorted(state["cluster_info"].items()) if cid != 0]
    rename_dropdown.options = options
    rename_dropdown.value = options[0] if options else None

def update_plot_traces():
    fw, labels, sources = state["fig_widget"], state["cluster_labels"], state["label_source"]
    if fw is None or labels is None:
        return
    visible = {label for label, cb in state["checkboxes"].items() if cb.value}
    mask = np.asarray([x in visible for x in sources], dtype=bool) if state["checkboxes"] else np.ones(len(labels), dtype=bool)
    state["filtered_mask"] = mask
    colors = [cluster_color_for(x) for x in labels]
    alphas = np.where(mask, ALPHA, ALPHA_INACTIVE)
    hover = [f"{src}<br>{cluster_name_for(cid)}<br>ID: {uid}" for src, cid, uid in zip(sources, labels, state["ids_umap"])]
    with fw.batch_update():
        for i in range(2):
            fw.data[i].text = hover
            if state["dim"] == "3D":
                fw.data[i].marker.color = [f"rgba({int(c[1:3],16)},{int(c[3:5],16)},{int(c[5:7],16)},{a})" for c, a in zip(colors, alphas)]
            else:
                fw.data[i].marker.color = colors
                fw.data[i].marker.opacity = alphas
        fw.data = tuple(fw.data[:2])
        for species in dict.fromkeys(sources.tolist()):
            count = int(np.sum(sources == species))
            if state["dim"] == "3D":
                fw.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers", marker=dict(size=8, color=DEFAULT_COLOR), name=f"{species} (n={count})"))
            else:
                fw.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=8, color=DEFAULT_COLOR), name=f"{species} (n={count})"))
        for cid in sorted((x for x in state["cluster_info"] if x != 0), key=lambda x: -int(np.sum(labels == x))):
            info, count = state["cluster_info"][cid], int(np.sum(labels == cid))
            if state["dim"] == "3D":
                fw.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers", marker=dict(size=8, color=info["color"]), name=f"{info['name']} (n={count})"))
            else:
                fw.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=8, color=info["color"]), name=f"{info['name']} (n={count})"))
    refresh_cluster_dropdown()

def update_visibility():
    update_plot_traces()

###################################### selection and editing
def _selection_callback(source):
    def callback(trace, points, selector):
        indices = list(getattr(points, "point_inds", []) or [])
        if indices:
            state["selected_indices"] = sorted(set(int(i) for i in indices if 0 <= int(i) < len(state["ids_umap"])))
            state["last_selection_source"] = source
            with output_area:
                clear_output(wait=True)
                print(f"Selected {len(state['selected_indices'])} points from {source.upper()}.")
    return callback

def _click_callback(source):
    def callback(trace, points, state_event):
        indices = list(getattr(points, "point_inds", []) or [])
        if indices:
            state["selected_indices"] = sorted(set(int(i) for i in indices if 0 <= int(i) < len(state["ids_umap"])))
            state["last_selection_source"] = source
            with output_area:
                clear_output(wait=True)
                print(f"Selected {len(state['selected_indices'])} point(s) from {source.upper()}.")
    return callback

def assign_selected(_):
    with output_area:
        clear_output(wait=True)
        indices = state["selected_indices"]
        if not indices or state["cluster_labels"] is None:
            print("Select points in either plot first.")
            return
        cid = next_cluster_id()
        prefix = "U" if state["last_selection_source"] == "umap" else "T"
        name = cluster_name.value.strip()
        if not name or name == "NewCluster":
            name = f"{prefix}-{cid:02d}"
        state["cluster_labels"][indices] = cid
        state["cluster_info"][cid] = {"name": name, "color": cluster_color.value}
        state["selected_indices"] = []
        update_plot_traces()
        print(f"Assigned {len(indices)} points to cluster {cid} '{name}'.")

def rename_cluster(_):
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("No cluster available to rename.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        name = rename_text.value.strip()
        if not name:
            print("Enter a new cluster name.")
            return
        state["cluster_info"][cid]["name"] = name
        update_plot_traces()
        print(f"Renamed cluster {cid} to '{name}'.")

def reset_clusters(_):
    with output_area:
        clear_output(wait=True)
        if state["ids_umap"] is None:
            print("Load data first.")
            return
        state["cluster_labels"] = np.zeros(len(state["ids_umap"]), dtype=int)
        state["cluster_info"] = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
        state["selected_indices"] = []
        update_plot_traces()
        print("All points reset to unassigned.")

def delete_cluster(_):
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("Choose a cluster first.")
            return
        cid = int(rename_dropdown.value.split(":", 1)[0])
        if cid == 0 or cid not in state["cluster_info"]:
            print("Select a valid cluster.")
            return
        state["cluster_labels"][state["cluster_labels"] == cid] = 0
        del state["cluster_info"][cid]
        update_plot_traces()
        print(f"Deleted cluster {cid}; its points are now unassigned.")

def reset_angles(_):
    fw = state["fig_widget"]
    if fw is None:
        return
    if state["dim"] == "3D":
        fw.update_layout(scene_camera=None, scene2_camera=None)
    else:
        fw.update_xaxes(autorange=True)
        fw.update_yaxes(autorange=True)

###################################### loading and plotting
def _display_title(basename):
    return basename.replace("_", " ")

def _build_figure():
    dim, u, t, ids, labels = state["dim"], state["umap_coords"], state["tsne_coords"], state["ids_umap"], state["label_source"]
    initial = [DEFAULT_COLOR] * len(ids)
    title = state["display_title"]
    if dim == "2D":
        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{title} - UMAP", f"{title} - t-SNE"), horizontal_spacing=0.08)
        fig.add_trace(go.Scattergl(x=u[:, 0], y=u[:, 1], mode="markers", marker=dict(size=MARKER_SIZE, color=initial, opacity=ALPHA), text=labels, customdata=ids, name="UMAP data", showlegend=False), row=1, col=1)
        fig.add_trace(go.Scattergl(x=t[:, 0], y=t[:, 1], mode="markers", marker=dict(size=MARKER_SIZE, color=initial, opacity=ALPHA), text=labels, customdata=ids, name="t-SNE data", showlegend=False), row=1, col=2)
        for col, coords, name, zoom in [(1, u, "UMAP", ZOOM_FACTOR_UMAP), (2, t, "t-SNE", ZOOM_FACTOR_TSNE)]:
            x0, x1 = float(coords[:, 0].min()), float(coords[:, 0].max())
            y0, y1 = float(coords[:, 1].min()), float(coords[:, 1].max())
            xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
            fig.update_xaxes(title_text=f"{name}-1", range=[xm-(x1-x0)*zoom/2, xm+(x1-x0)*zoom/2], tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY), ticks="outside", linecolor=LINE_COLOR, linewidth=LINE_WIDTH, showgrid=False, zeroline=False, row=1, col=col)
            fig.update_yaxes(title_text=f"{name}-2", range=[ym-(y1-y0)*zoom/2, ym+(y1-y0)*zoom/2], tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY), ticks="outside", linecolor=LINE_COLOR, linewidth=LINE_WIDTH, showgrid=False, zeroline=False, row=1, col=col)
    else:
        fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]], subplot_titles=(f"{title} - UMAP", f"{title} - t-SNE"), horizontal_spacing=0.04)
        fig.add_trace(go.Scatter3d(x=u[:, 0], y=u[:, 1], z=u[:, 2], mode="markers", marker=dict(size=MARKER_SIZE_3D, color=initial, opacity=ALPHA), text=labels, customdata=ids, name="UMAP data", showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter3d(x=t[:, 0], y=t[:, 1], z=t[:, 2], mode="markers", marker=dict(size=MARKER_SIZE_3D, color=initial, opacity=ALPHA), text=labels, customdata=ids, name="t-SNE data", showlegend=False), row=1, col=2)
        axis = dict(backgroundcolor="white", showbackground=True, showline=True, linecolor=LINE_COLOR, linewidth=LINE_WIDTH, gridcolor="#DDDDDD", zerolinecolor="#BBBBBB", tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="black"))
        fig.update_layout(scene=dict(xaxis=dict(title="UMAP-1", **axis), yaxis=dict(title="UMAP-2", **axis), zaxis=dict(title="UMAP-3", **axis)), scene2=dict(xaxis=dict(title="t-SNE-1", **axis), yaxis=dict(title="t-SNE-2", **axis), zaxis=dict(title="t-SNE-3", **axis)))
    fig.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT, dragmode="lasso" if dim == "2D" else "orbit", hovermode="closest", plot_bgcolor="white", paper_bgcolor="white", margin=dict(l=35, r=30, t=70, b=100), legend=dict(orientation="h", y=-0.16, x=0.5, xanchor="center", yanchor="top", bgcolor="rgba(0,0,0,0)"), font=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="#222222"))
    fig.update_annotations(font=dict(size=TITLE_FONT_SIZE, family=TICK_FONT_FAMILY, color="#111111"))
    return fig

def apply_color_source(_=None):
    if state["ids_umap"] is None or color_source_dropdown.value == "(Self)":
        return
    selected = color_source_dropdown.value
    basename, dim = selected.rsplit("_", 1)
    with output_area:
        clear_output(wait=True)
        if dim != state["dim"]:
            print(f"Color source must use the same dimensionality ({state['dim']}) as the current view.")
            return
        try:
            source_ids, source_coords = load_npz(basename, fit_on_dropdown.value, dim)
            source_labels, n_clusters = fit_hdbscan_auto(source_coords, mcs_slider.value)
            source_map = dict(zip(source_ids.tolist(), source_labels.tolist()))
            projected = np.asarray([source_map.get(uid, 0) for uid in state["ids_umap"].tolist()], dtype=int)
            info = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
            prefix = "U" if fit_on_dropdown.value == "umap" else "T"
            for cid in range(1, n_clusters + 1):
                info[cid] = {"name": f"{prefix}-{cid:02d}", "color": PALETTE_NATURE[(cid-1) % len(PALETTE_NATURE)]}
            state["cluster_labels"], state["cluster_info"] = projected, info
            update_plot_traces()
            print(f"Projected labels from '{selected}' onto current IDs; source clusters: {n_clusters}.")
        except Exception as exc:
            print(f"Could not project clusters from '{selected}': {exc}")

def load_and_plot(change=None):
    selected = change["new"] if isinstance(change, dict) and "new" in change else select_dropdown.value
    with output_area:
        clear_output(wait=True)
        if not selected:
            print("Select a dataset.")
            return
        basename, dim = selected.rsplit("_", 1)
        print(f"Loading '{basename}' ({dim})...")
        try:
            ids, u = load_npz(basename, "umap", dim)
            tsne_ids, t = load_npz(basename, "tsne", dim)
            t = align_embedding(ids, tsne_ids, t, f"{basename} t-SNE")
            sources = np.full(len(ids), basename, dtype=object)
            checklist_labels = [basename]
            state.update(ids_umap=ids, umap_coords=u, tsne_coords=t, label_source=sources, current_select=basename, dim=dim, display_title=_display_title(basename), selected_indices=[], cluster_labels=np.zeros(len(ids), dtype=int), cluster_info={0: {"name": "unassigned", "color": DEFAULT_COLOR}}, filtered_mask=np.ones(len(ids), dtype=bool))
            update_checklist(checklist_labels)
            try:
                fw = go.FigureWidget(_build_figure())
            except Exception as exc:
                raise RuntimeError("FigureWidget could not initialize. In JupyterLab run `pip install -U plotly anywidget ipywidgets`, restart the kernel, and reload the notebook.") from exc
            fw.data[0].on_selection(_selection_callback("umap"))
            fw.data[1].on_selection(_selection_callback("tsne"))
            fw.data[0].on_click(_click_callback("umap"))
            fw.data[1].on_click(_click_callback("tsne"))
            state["fig_widget"] = fw
            plot_container.children = [fw]
            update_plot_traces()
            if color_source_dropdown.value != "(Self)":
                apply_color_source()
            print(f"Loaded {len(ids):,} points for '{state['display_title']}' ({dim}). Use lasso/box selection where supported; clicks also select points.")
        except Exception as exc:
            print(f"Error: {exc}")
            traceback.print_exc()

###################################### exports
def _output_directory():
    raw = plot_fp_text.value.strip()
    if not raw:
        raise ValueError("Specify an export directory.")
    path = Path(raw) if Path(raw).is_absolute() else here() / raw
    path.mkdir(parents=True, exist_ok=True)
    return path

def export_results(_=None):
    with output_area:
        clear_output(wait=True)
        if state["ids_umap"] is None or state["cluster_labels"] is None:
            print("Load a dataset first.")
            return
        try:
            selected = np.zeros(len(state["ids_umap"]), dtype=bool)
            if state["selected_indices"]:
                selected[state["selected_indices"]] = True
            else:
                selected = state["filtered_mask"] if state["filtered_mask"] is not None else np.ones(len(selected), dtype=bool)
            df = pd.DataFrame({"id": state["ids_umap"], "cluster_id": state["cluster_labels"], "cluster_name": [cluster_name_for(x) for x in state["cluster_labels"]], "select": state["current_select"], "label": state["label_source"], "in_selection": selected})
            path = _output_directory() / f"{state['current_select']}_clusters_{state['dim']}.tsv"
            df.to_csv(path, sep="\t", index=False)
            print(f"Exported {len(df):,} rows to {path}")
        except Exception as exc:
            print(f"Export failed: {exc}")

def export_plot(_=None):
    with output_area:
        clear_output(wait=True)
        if state["fig_widget"] is None:
            print("Load a dataset first.")
            return
        try:
            out_dir, basename, dim = _output_directory(), state["current_select"], state["dim"]
            labels = state["cluster_labels"]
            colors = [cluster_color_for(x) for x in labels]
            if dim == "2D":
                fig, axes = plt.subplots(1, 2, figsize=(SQUARE_SIZE, SQUARE_SIZE * 0.5), dpi=DPI)
                for ax, coords, suffix, zoom in zip(axes, [state["umap_coords"], state["tsne_coords"]], ["UMAP", "t-SNE"], [ZOOM_FACTOR_UMAP, ZOOM_FACTOR_TSNE]):
                    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=MARKER_SIZE, alpha=ALPHA, linewidths=0, rasterized=True)
                    x0, x1, y0, y1 = coords[:, 0].min(), coords[:, 0].max(), coords[:, 1].min(), coords[:, 1].max()
                    xm, ym = (x0+x1)/2, (y0+y1)/2
                    ax.set_xlim(xm-(x1-x0)*zoom/2, xm+(x1-x0)*zoom/2)
                    ax.set_ylim(ym-(y1-y0)*zoom/2, ym+(y1-y0)*zoom/2)
                    ax.set_title(f"{state['display_title']} - {suffix}", fontsize=TITLE_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                    ax.set_xlabel(f"{suffix}-1", fontsize=TICK_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                    ax.set_ylabel(f"{suffix}-2", fontsize=TICK_FONT_SIZE, fontname=TICK_FONT_FAMILY)
                    ax.tick_params(axis="both", labelsize=TICK_FONT_SIZE, length=4, width=LINE_WIDTH, colors="black")
                    for spine in ax.spines.values():
                        spine.set_color(LINE_COLOR); spine.set_linewidth(LINE_WIDTH)
                    ax.set_facecolor("white"); ax.grid(False)
                fig.patch.set_facecolor("white"); plt.tight_layout()
                for ext in ("pdf", "png"):
                    fig.savefig(out_dir / f"{basename}_plot_{dim}.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
                plt.close(fig)
            else:
                fig = go.Figure(state["fig_widget"].to_dict())
                fig.update_layout(width=1500, height=900)
                for ext in ("pdf", "png"):
                    try:
                        fig.write_image(str(out_dir / f"{basename}_plot_{dim}.{ext}"), scale=2 if ext == "png" else 1)
                    except Exception as exc:
                        raise RuntimeError("Static 3D export needs Kaleido: `pip install -U kaleido`.") from exc
            handles = [Line2D([0], [0], marker="o", linestyle="", color=info["color"], label=f"{info['name']} (n={int(np.sum(labels == cid))})", markersize=5) for cid, info in sorted(state["cluster_info"].items()) if cid != 0]
            handles.append(Line2D([0], [0], marker="o", linestyle="", color=DEFAULT_COLOR, label=f"Unassigned (n={int(np.sum(labels == 0))})", markersize=5))
            for orientation in ("horizontal", "vertical"):
                legend_fig = plt.figure(figsize=(10, 1.8) if orientation == "horizontal" else (4, max(2, len(handles)*0.35)), dpi=DPI)
                legend_fig.legend(handles=handles, loc="center", ncol=LEGEND_NCOL if orientation == "horizontal" else 1, frameon=False, fontsize=LEGEND_FONT_SIZE)
                for ext in ("pdf", "png"):
                    legend_fig.savefig(out_dir / f"{basename}_legend_{orientation}.{ext}", dpi=DPI, bbox_inches="tight", facecolor="white")
                plt.close(legend_fig)
            print(f"Saved plot and legends to {out_dir}")
        except Exception as exc:
            print(f"Plot export failed: {exc}")

###################################### batch processing
def run_all(_=None):
    with output_area:
        clear_output(wait=True)
        try:
            out_dir = _output_directory()
        except Exception as exc:
            print(exc); return
        failed, completed = [], 0
        print(f"Batch clustering {len(valid_options_all)} dataset(s)...")
        for selected in valid_options_all:
            try:
                basename, dim = selected.rsplit("_", 1)
                ids, coords = load_npz(basename, fit_on_dropdown.value, dim)
                labels, n_clusters = fit_hdbscan_auto(coords, mcs_slider.value)
                info = {0: {"name": "unassigned"}}
                prefix = "U" if fit_on_dropdown.value == "umap" else "T"
                for cid in range(1, n_clusters+1): info[cid] = {"name": f"{prefix}-{cid:02d}"}
                pd.DataFrame({"id": ids, "cluster_id": labels, "cluster_name": [info[int(x)]["name"] for x in labels], "select": basename, "dim": dim}).to_csv(out_dir / f"{basename}_clusters_{dim}.tsv", sep="\t", index=False)
                completed += 1
                print(f"[{completed}/{len(valid_options_all)}] {selected}: {n_clusters} clusters")
            except Exception as exc:
                failed.append((selected, str(exc))); print(f"[failed] {selected}: {exc}")
        print(f"Batch complete: {completed}/{len(valid_options_all)} succeeded.")
        if failed:
            print("Failures:"); [print(f"  {name}: {error}") for name, error in failed]

###################################### bindings and display
select_dropdown.observe(load_and_plot, names="value")
color_source_dropdown.observe(lambda change: apply_color_source() if change["new"] != "(Self)" else None, names="value")
reset_angles_btn.on_click(reset_angles)
fit_btn.on_click(lambda _: fit_clusters())
assign_btn.on_click(assign_selected)
rename_btn.on_click(rename_cluster)
reset_btn.on_click(reset_clusters)
delete_btn.on_click(delete_cluster)
export_tsv_btn.on_click(export_results)
export_plot_btn.on_click(export_plot)
run_all_btn.on_click(run_all)

def fit_clusters():
    with output_area:
        clear_output(wait=True)
        if state["ids_umap"] is None:
            print("Load data first."); return
        try:
            if color_source_dropdown.value != "(Self)":
                apply_color_source(); return
            method = fit_on_dropdown.value
            coords = state["umap_coords"] if method == "umap" else state["tsne_coords"]
            print(f"Fitting HDBSCAN on {method.upper()} with MinCS={mcs_slider.value}...")
            labels, n_clusters = fit_hdbscan_auto(coords, mcs_slider.value)
            state["cluster_labels"] = labels
            state["cluster_info"] = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
            prefix = "U" if method == "umap" else "T"
            for cid in range(1, n_clusters+1): state["cluster_info"][cid] = {"name": f"{prefix}-{cid:02d}", "color": PALETTE_NATURE[(cid-1) % len(PALETTE_NATURE)]}
            update_plot_traces()
            print(f"{method.upper()}: {n_clusters} cluster(s) found.")
        except Exception as exc:
            print(f"Clustering failed: {exc}")

display(VBox([HBox([select_dropdown, color_source_dropdown, reset_angles_btn]), checklist_box, HBox([mcs_slider, fit_on_dropdown, fit_btn]), HBox([cluster_name, cluster_color, assign_btn]), HBox([rename_dropdown, rename_text, rename_btn, delete_btn, reset_btn]), HBox([plot_fp_text, export_tsv_btn, export_plot_btn, run_all_btn]), plot_container, output_area]))

###################################### final execution log
elapsed = time.time() - start_time
try:
    peak_memory_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 if os.name != "darwin" else 1024*1024)
except Exception:
    peak_memory_mb = float("nan")
print(f"[PYTHON-INFO] elapsed={elapsed:.2f}s | peak_memory={peak_memory_mb:.1f}MB | datasets={len(valid_options_all)}")


FileNotFoundError: Could not find any directory containing paired UMAP/t-SNE embedding NPZ files. Set DIMRED_BASE_DIR to the exact directory containing files named like <dataset>_2D_umap.npz and <dataset>_2D_tsne.npz. Searched beneath project root /orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian and notebook working directory /orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/nbs. No configuration files were accessed.

In [4]:
from pathlib import Path

roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "data",
    Path.cwd().parent / "data",
]

seen = set()

for root in roots:
    if not root.is_dir():
        continue

    for path in root.rglob("*.npz"):
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            print(resolved)

if not seen:
    print("Nessun file .npz trovato nelle directory esaminate.")

/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Mammalia_tsne_nc3_perp30.0_iter1000_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Sauropsis_tsne_nc3_perp30.0_iter1000_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Mammalia_tsne_nc2_perp30.0_iter1000_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Sauropsis_umap_nc2_nn15_mindist0.1_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Mammalia_umap_nc3_nn15_mindist0.1_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Mammalia_Sauropsis_umap_nc2_nn15_mindist0.1_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Sauropsis_tsne_nc2_perp30.0_iter1000_weights.npz
/orfeo/cephfs/scratch/area/ssenci/senci_projects/REPR_LRN/SCambrian/data/dimred/Sauropsis_umap_nc